# Prophet-Style Simulation — Shredder Bearing Temperature Prediction

## 개요

이 노트북은 Facebook Prophet의 **핵심 원리**를 체험하는 시뮬레이션입니다.

| 항목 | 내용 |
|------|------|
| **모델** | Prophet (Meta/Facebook, 2017) |
| **시나리오** | 슈레더 베어링 온도 90일 데이터로 미래 예측 |
| **핵심 원리** | 시계열 = 트렌드 + 계절성 + 노이즈 분해 |

### Prophet의 수학적 모델

```
y(t) = g(t) + s(t) + h(t) + ε(t)
       트렌드  계절성  휴일   노이즈

이 시뮬레이션에서:
  g(t) = 칼날 마모에 의한 온도 상승 트렌드 (하루 +0.03°C)
  s(t) = 일간 패턴 (낮 가동→온도↑, 밤 정지→온도↓)
       + 주간 패턴 (주말 비가동→온도↓)
  ε(t) = 센서 노이즈 + 이상 고온 이벤트
```

> **참고**: Prophet 라이브러리 대신 `statsmodels STL 분해 + LinearRegression`으로 Prophet의 핵심 원리를 동일하게 구현합니다.

---
## Step 0. 라이브러리 설치 및 Import

In [ ]:
!pip install -q statsmodels scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

print('All libraries loaded successfully!')

---
## Step 1. Shredder Sensor Data Generation

산업용 슈레더(Shredder)의 센서 데이터를 시뮬레이션합니다.

### 생성되는 센서 데이터

| 센서 | 단위 | 정상 범위 | 설명 |
|------|------|-----------|------|
| temperature | °C | 20~38 | 베어링 온도 (예측 대상) |
| vibration | mm/s | 1.5~4.0 | 진동 RMS |
| current | A | 65~110 | 모터 전류 |
| rpm | rpm | 19~21 | 회전 속도 |
| throughput | t/h | 1.5~3.5 | 처리량 |

### 데이터에 포함된 패턴

- **일간 패턴**: 낮에 가동 → 온도 상승, 밤에 정지 → 온도 하강
- **주간 패턴**: 주말 비가동 → 온도/전류 하강
- **마모 트렌드**: 칼날 마모로 90일간 온도가 서서히 상승 (+2.7°C)
- **이상 이벤트**: 0.5% 확률로 고온 스파이크 발생 (+15~30°C)

In [ ]:
def generate_shredder_data(days=90, freq_minutes=10, seed=42):
    """
    Shredder sensor data simulator.
    Generates realistic bearing temperature, vibration, current, rpm, throughput.
    """
    np.random.seed(seed)

    n_points = days * 24 * 60 // freq_minutes
    timestamps = pd.date_range(
        start='2026-01-01',
        periods=n_points,
        freq=f'{freq_minutes}min'
    )

    t = np.arange(n_points)
    hours = np.array([ts.hour for ts in timestamps])
    dow = np.array([ts.dayofweek for ts in timestamps])

    # === Bearing Temperature ===
    base_temp = 28.0
    daily_pattern = 5.0 * np.sin(2 * np.pi * hours / 24 - np.pi/2)
    weekly_pattern = np.where(dow >= 5, -3.0, 0.0)
    wear_trend = 0.03 * t / (24 * 60 / freq_minutes)
    noise = np.random.normal(0, 0.8, n_points)

    anomaly_mask = np.random.random(n_points) < 0.005
    anomaly_spike = anomaly_mask * np.random.uniform(15, 30, n_points)

    temperature = base_temp + daily_pattern + weekly_pattern + wear_trend + noise + anomaly_spike
    temperature = np.clip(temperature, 15, 80)

    # === Vibration RMS ===
    base_vib = 2.5
    vib_daily = 0.5 * np.sin(2 * np.pi * hours / 24)
    vib_wear = 0.02 * t / (24 * 60 / freq_minutes)
    vib_noise = np.random.normal(0, 0.3, n_points)
    vib_anomaly = anomaly_mask * np.random.uniform(5, 15, n_points)

    vibration = base_vib + vib_daily + vib_wear + vib_noise + vib_anomaly
    vibration = np.clip(vibration, 0.5, 25)

    # === Motor Current ===
    base_cur = 85.0
    cur_daily = 10.0 * np.sin(2 * np.pi * hours / 24 - np.pi/3)
    cur_wear = 0.05 * t / (24 * 60 / freq_minutes)
    cur_noise = np.random.normal(0, 2.0, n_points)
    cur_weekend = np.where(dow >= 5, -30.0, 0.0)

    current = base_cur + cur_daily + cur_wear + cur_noise + cur_weekend
    current = np.clip(current, 20, 150)

    # === RPM ===
    base_rpm = 20.0
    rpm_var = np.random.normal(0, 0.3, n_points)
    rpm_weekend = np.where(dow >= 5, -15.0, 0.0)

    rpm = base_rpm + rpm_var + rpm_weekend
    rpm = np.clip(rpm, 0, 25)

    # === Throughput ===
    base_tp = 2.5
    tp_daily = 0.5 * np.sin(2 * np.pi * hours / 24 - np.pi/4)
    tp_noise = np.random.normal(0, 0.15, n_points)
    tp_weekend = np.where(dow >= 5, -2.0, 0.0)

    throughput = base_tp + tp_daily + tp_noise + tp_weekend
    throughput = np.clip(throughput, 0, 4)

    df = pd.DataFrame({
        'timestamp': timestamps,
        'temperature': np.round(temperature, 2),
        'vibration': np.round(vibration, 2),
        'current': np.round(current, 2),
        'rpm': np.round(rpm, 2),
        'throughput': np.round(throughput, 2)
    })

    return df

print('generate_shredder_data() defined.')

In [ ]:
# Generate 90 days of data, sampled every 1 hour (for Prophet-style analysis)
df = generate_shredder_data(days=90, freq_minutes=60)

print(f'Generated data: {len(df)} samples')
print(f'Period: {df["timestamp"].min()} ~ {df["timestamp"].max()}')
print(f'\nStatistics:')
df.describe().round(2)

---
## Step 2. Raw Data Visualization

생성된 원본 데이터를 확인합니다. 각 센서의 시간에 따른 변화를 관찰하세요.

**관찰 포인트:**
- 온도(temperature)에 일간/주간 패턴이 보이는가?
- 90일간 온도가 서서히 상승하는 트렌드가 보이는가?
- 간헐적으로 튀는 이상값(스파이크)이 보이는가?

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)

sensors = [
    ('temperature', 'Bearing Temperature', 'C', 'tab:red'),
    ('vibration', 'Vibration RMS', 'mm/s', 'tab:blue'),
    ('current', 'Motor Current', 'A', 'tab:green'),
    ('rpm', 'RPM', 'rpm', 'tab:orange'),
    ('throughput', 'Throughput', 't/h', 'tab:purple'),
]

for ax, (col, title, unit, color) in zip(axes, sensors):
    ax.plot(df['timestamp'], df[col], color=color, alpha=0.7, linewidth=0.5)
    ax.set_ylabel(f'{title} ({unit})', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_title(f'{title}', fontsize=12, fontweight='bold')

axes[-1].set_xlabel('Date', fontsize=11)
fig.suptitle('Shredder Sensor Data — 90 Days Overview', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 온도 데이터 확대 (처음 7일)

처음 7일만 확대하여 일간 패턴(낮↑/밤↓)과 주말 하락을 명확히 확인합니다.

In [ ]:
# Zoom in: first 7 days
first_week = df[df['timestamp'] < '2026-01-08']

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(first_week['timestamp'], first_week['temperature'], 'b-o', markersize=3, linewidth=1)
ax.set_title('Temperature — First 7 Days (Daily Pattern Visible)', fontsize=13, fontweight='bold')
ax.set_ylabel('Temperature (C)')
ax.set_xlabel('Date')
ax.grid(True, alpha=0.3)

# Mark weekend
for d in pd.date_range('2026-01-03', '2026-01-04'):
    ax.axvspan(d, d + pd.Timedelta(days=1), alpha=0.1, color='gray', label='Weekend' if d.day == 3 else '')

ax.legend()
plt.tight_layout()
plt.show()

---
## Step 3. Train/Test Split

시계열 데이터는 **반드시 시간순으로 분할**해야 합니다. 랜덤 분할은 미래 정보가 학습에 포함되어 성능이 과대 추정됩니다.

```
|<--- Train (70%) --->|<--- Test (30%) --->|
|     Jan ~ early Mar  |   mid Mar ~ end Mar |
```

In [ ]:
# Extract temperature time series
ts = df.set_index('timestamp')['temperature']

# Time-ordered split (no random shuffle!)
train_size = int(len(ts) * 0.7)
train_ts = ts.iloc[:train_size]
test_ts = ts.iloc[train_size:]

print(f'Train: {len(train_ts)} samples ({train_ts.index.min().date()} ~ {train_ts.index.max().date()})')
print(f'Test : {len(test_ts)} samples ({test_ts.index.min().date()} ~ {test_ts.index.max().date()})')

In [ ]:
# Visualize train/test split
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train_ts.index, train_ts.values, 'b.', alpha=0.3, markersize=2, label='Train')
ax.plot(test_ts.index, test_ts.values, 'orange', alpha=0.5, markersize=2, label='Test', marker='.', linestyle='none')
ax.axvline(x=train_ts.index[-1], color='green', linestyle='--', linewidth=2, label='Split boundary')
ax.set_title('Train / Test Split (70% / 30%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Temperature (C)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 4. STL Decomposition (Prophet Core Principle)

Prophet의 핵심은 시계열을 **3가지 구성요소로 분해**하는 것입니다:

$$y(t) = \text{Trend}(t) + \text{Seasonality}(t) + \text{Residual}(t)$$

- **Trend**: 장기적 추세 (칼날 마모 → 온도 점진 상승)
- **Seasonality**: 반복 패턴 (낮에 높고 밤에 낮은 24시간 주기)
- **Residual**: 나머지 (센서 노이즈 + 이상 이벤트)

여기서는 STL(Seasonal-Trend decomposition using LOESS)을 사용합니다.

In [ ]:
# STL decomposition with 24-hour period
stl = STL(train_ts, period=24, robust=True)
result = stl.fit()

print('STL Decomposition Complete!')
print(f'  Trend range   : {result.trend.min():.1f} ~ {result.trend.max():.1f} C')
print(f'  Seasonal range: {result.seasonal.min():.1f} ~ {result.seasonal.max():.1f} C')
print(f'  Residual std  : {result.resid.std():.2f} C')

### 4-1. Decomposition Visualization

분해된 3가지 구성요소를 개별적으로 시각화합니다.

In [ ]:
# Original data
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(train_ts.index, train_ts.values, 'b-', alpha=0.5, linewidth=0.5)
ax.set_title('Original: Bearing Temperature (Train)', fontsize=13, fontweight='bold')
ax.set_ylabel('Temperature (C)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Trend component
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(train_ts.index, result.trend.values, 'g-', linewidth=2)
ax.set_title('Trend: Blade Wear -> Gradual Temperature Rise (+0.03 C/day)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Trend (C)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Trend increase over train period: '
      f'{result.trend.values[-1] - result.trend.values[0]:.2f} C')

In [ ]:
# Seasonal component (24-hour pattern)
seasonal_pattern = result.seasonal.values[-24:]

fig, ax = plt.subplots(figsize=(10, 4))
hours = np.arange(24)
colors = ['#ff6b6b' if v > 0 else '#4dabf7' for v in seasonal_pattern]
ax.bar(hours, seasonal_pattern, color=colors, alpha=0.8, edgecolor='white')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_title('Seasonality: 24h Daily Pattern (Day=High, Night=Low)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Hour (0-23)')
ax.set_ylabel('Seasonal Effect (C)')
ax.set_xticks(range(0, 24))
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'Peak hour: {hours[np.argmax(seasonal_pattern)]}:00 ({seasonal_pattern.max():+.2f} C)')
print(f'Low  hour: {hours[np.argmin(seasonal_pattern)]}:00 ({seasonal_pattern.min():+.2f} C)')

In [ ]:
# Residual (noise + anomalies)
noise_std = result.resid.std()

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(train_ts.index, result.resid.values, 'purple', alpha=0.5, linewidth=0.5)
ax.axhline(y=0, color='black', linewidth=1)
ax.axhline(y=2*noise_std, color='red', linestyle='--', alpha=0.5,
           label=f'+/- 2 sigma ({2*noise_std:.1f} C)')
ax.axhline(y=-2*noise_std, color='red', linestyle='--', alpha=0.5)
ax.set_title('Residual: Noise + Anomaly Spikes (after removing Trend & Seasonality)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Residual (C)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

anomalies = np.abs(result.resid.values) > 2 * noise_std
print(f'Residual std: {noise_std:.2f} C')
print(f'Anomaly points (|residual| > 2 sigma): {anomalies.sum()} / {len(result.resid)}')

---
## Step 5. Forecasting (Trend Extrapolation + Seasonal Repeat)

Prophet의 예측 방식:

1. **트렌드 외삽**: 학습된 트렌드를 선형 회귀로 미래에 연장
2. **계절성 반복**: 학습된 24시간 패턴을 반복 적용
3. **불확실성 구간**: 잔차의 표준편차(σ)로 ±2σ 구간 제공

```
Forecast = Trend_extrapolation + Seasonal_repeat
Uncertainty = ± 2 × residual_std
```

In [ ]:
# Trend extrapolation using Linear Regression
X_train_idx = np.arange(len(train_ts)).reshape(-1, 1)
trend_model = LinearRegression()
trend_model.fit(X_train_idx, result.trend.values)

X_test_idx = np.arange(len(train_ts), len(train_ts) + len(test_ts)).reshape(-1, 1)
trend_pred = trend_model.predict(X_test_idx)

# Seasonal pattern repeat
seasonal_pred = np.tile(seasonal_pattern, len(test_ts) // 24 + 1)[:len(test_ts)]

# Final forecast = Trend + Seasonality
forecast = trend_pred + seasonal_pred

print(f'Trend slope : {trend_model.coef_[0]:.4f} C/hour ({trend_model.coef_[0]*24:.3f} C/day)')
print(f'Forecast length: {len(forecast)} hours ({len(forecast)/24:.0f} days)')

---
## Step 6. Performance Evaluation

테스트 구간의 실제값과 예측값을 비교합니다.

| 지표 | 의미 |
|------|------|
| **MAE** | 평균 절대 오차 — 평균적으로 몇 도 틀리는가 |
| **RMSE** | 평균 제곱근 오차 — 큰 오차에 더 민감 |

In [ ]:
mae = mean_absolute_error(test_ts.values, forecast)
rmse = np.sqrt(np.mean((test_ts.values - forecast)**2))

print('=' * 50)
print('  Performance Evaluation')
print('=' * 50)
print(f'  MAE  = {mae:.2f} C (average error: {mae:.2f} degrees)')
print(f'  RMSE = {rmse:.2f} C')
print('=' * 50)

---
## Step 7. Result Visualization

### 7-1. Full Forecast vs Actual

전체 기간의 실제 온도와 테스트 구간의 예측값을 비교합니다.
분홍색 영역은 **불확실성 구간(±2σ)**으로, 실제값이 이 안에 들어올 확률이 약 95%입니다.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(ts.index, ts.values, 'b.', alpha=0.3, markersize=2, label='Actual Temperature')
ax.plot(test_ts.index, forecast, 'r-', linewidth=1.5, label='Prophet-Style Forecast')
ax.fill_between(test_ts.index,
                forecast - 2*noise_std,
                forecast + 2*noise_std,
                alpha=0.15, color='red', label=f'Uncertainty (+/- 2 sigma = {2*noise_std:.1f} C)')
ax.axvline(x=train_ts.index[-1], color='green', linestyle='--', linewidth=2, label='Train/Test boundary')

ax.set_title(f'Prophet-Style Forecast — Shredder Bearing Temperature (MAE={mae:.2f} C)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Temperature (C)')
ax.set_xlabel('Date')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7-2. Trend Extrapolation

학습 구간에서 감지된 트렌드(초록선)가 테스트 구간으로 외삽(빨간 점선)됩니다.
칼날 마모에 의해 온도가 하루 약 +0.03°C씩 상승하는 것을 포착합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(train_ts.index, result.trend.values, 'g-', linewidth=2, label='Train Trend')
ax.plot(test_ts.index, trend_pred, 'r--', linewidth=2, label='Trend Extrapolation (Forecast)')
ax.axvline(x=train_ts.index[-1], color='gray', linestyle=':', linewidth=1)

ax.set_title('Prophet Core 1: Trend Detection (Blade Wear -> Temperature Rise)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Trend (C)')
ax.set_xlabel('Date')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7-3. Test Period Close-up

테스트 구간의 처음 5일을 확대하여 예측의 정밀도를 확인합니다.

In [ ]:
# Close-up: first 5 days of test period
n_closeup = 5 * 24  # 5 days

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_ts.index[:n_closeup], test_ts.values[:n_closeup],
        'b-o', markersize=3, linewidth=1, alpha=0.7, label='Actual')
ax.plot(test_ts.index[:n_closeup], forecast[:n_closeup],
        'r-s', markersize=3, linewidth=1, alpha=0.7, label='Forecast')
ax.fill_between(test_ts.index[:n_closeup],
                forecast[:n_closeup] - 2*noise_std,
                forecast[:n_closeup] + 2*noise_std,
                alpha=0.15, color='red', label='Uncertainty (+/- 2 sigma)')

ax.set_title('Test Period Close-up (First 5 Days)', fontsize=13, fontweight='bold')
ax.set_ylabel('Temperature (C)')
ax.set_xlabel('Date')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7-4. Error Distribution

예측 오차(실제 - 예측)의 분포를 히스토그램으로 확인합니다. 정규분포에 가까우면 모델이 체계적 편향 없이 잘 작동하고 있다는 의미입니다.

In [ ]:
errors = test_ts.values - forecast

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(errors, bins=50, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(x=0, color='red', linewidth=2, linestyle='--', label='Zero error')
ax.axvline(x=np.mean(errors), color='orange', linewidth=2, label=f'Mean error: {np.mean(errors):.2f} C')

ax.set_title('Forecast Error Distribution (Actual - Predicted)', fontsize=13, fontweight='bold')
ax.set_xlabel('Error (C)')
ax.set_ylabel('Density')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Error statistics:')
print(f'  Mean  : {np.mean(errors):+.2f} C')
print(f'  Std   : {np.std(errors):.2f} C')
print(f'  Median: {np.median(errors):+.2f} C')
print(f'  Within +/- 2 C: {(np.abs(errors) < 2).mean()*100:.1f}%')

---
## Step 8. Summary

### Prophet 핵심 특징 요약

| 구성요소 | 이 시뮬레이션에서 | 슈레더 현장 의미 |
|----------|-------------------|------------------|
| **Trend** | 90일간 온도 점진 상승 | 칼날 마모 → 마찰열 증가 |
| **Seasonality** | 24시간 주기 반복 | 낮 가동→온도↑, 밤 정지→온도↓ |
| **Residual** | 노이즈 + 스파이크 | 센서 오차 + 이상 이벤트 |

### 장점
- 트렌드/계절성을 **자동 분리** → "왜 올라가는지" 설명 가능
- **불확실성 구간** 제공 (±2σ)
- 코드가 매우 간단 (실제 Prophet은 3줄이면 완성)

### 단점
- **단변량만 가능** — 온도 하나만 사용 (진동, 전류 동시 활용 불가)
- 비선형 복잡 패턴에 약함

### 슈레더 적용 권장
- **적합**: 처리량/에너지 장기 트렌드 예측
- **부적합**: 베어링 이상 탐지 (다변량 필요 → LSTM, TFT 사용)

---

> **다음 단계**: `02_LSTM` 폴더에서 다변량 딥러닝 시계열 예측을 체험해 보세요.

In [ ]:
print('=' * 60)
print('  Prophet-Style Simulation Complete!')
print('=' * 60)
print(f'''
  y(t) = Trend + Seasonality + Noise

  1. Trend    : Temperature rising {result.trend.values[-1] - result.trend.values[0]:.1f} C over train period
                (blade wear -> friction increase)

  2. Seasonal : +/- {abs(seasonal_pattern).max():.1f} C daily pattern
                (daytime operation -> temp up, nighttime stop -> temp down)

  3. Residual : std = {noise_std:.2f} C
                (sensor noise + anomaly spikes)

  Performance:
    MAE  = {mae:.2f} C
    RMSE = {rmse:.2f} C
''')